# 05 Governance Risk Scoring Demo

This notebook demonstrates system inventory, governance evidence validation, and residual risk scoring.


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

REPO_ROOT = Path.cwd()
PROCESSED_DIR = REPO_ROOT / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

## AI System Inventory

Every AI system should be registered before deployment or scaling.

In [ ]:
inventory = pd.DataFrame(
    [
        {
            "system_id": "AI-001",
            "system_name": "Academic Integrity AI Detector",
            "system_type": "ai_detection_tool",
            "deployment_status": "Live",
            "owner_team": "Academic Registry",
            "vendor_type": "Third-party",
            "student_impact_area": "Academic misconduct",
            "risk_tier": "High",
            "human_review_required": True,
        },
        {
            "system_id": "AI-002",
            "system_name": "Student Support Chatbot",
            "system_type": "student_chatbot",
            "deployment_status": "Live",
            "owner_team": "Student Services",
            "vendor_type": "Third-party",
            "student_impact_area": "Student support",
            "risk_tier": "Moderate",
            "human_review_required": True,
        },
        {
            "system_id": "AI-003",
            "system_name": "VLE Engagement Analytics Dashboard",
            "system_type": "vle_engagement_analytics",
            "deployment_status": "Live",
            "owner_team": "Learning Analytics",
            "vendor_type": "Platform-integrated",
            "student_impact_area": "Learning support",
            "risk_tier": "Moderate",
            "human_review_required": True,
        },
        {
            "system_id": "AI-004",
            "system_name": "Career Services CV Reviewer",
            "system_type": "career_service_ai",
            "deployment_status": "Live",
            "owner_team": "Careers Service",
            "vendor_type": "Third-party",
            "student_impact_area": "Employability",
            "risk_tier": "Moderate",
            "human_review_required": False,
        },
        {
            "system_id": "AI-005",
            "system_name": "AI-Assisted Feedback Tool",
            "system_type": "ai_assisted_marking_feedback",
            "deployment_status": "Pilot",
            "owner_team": "Assessment Innovation",
            "vendor_type": "Third-party",
            "student_impact_area": "Assessment feedback",
            "risk_tier": "High",
            "human_review_required": True,
        },
        {
            "system_id": "AI-006",
            "system_name": "Early Alert Prediction Model",
            "system_type": "early_alert_prediction",
            "deployment_status": "Proposed",
            "owner_team": "Student Success",
            "vendor_type": "Internal / third-party hybrid",
            "student_impact_area": "Progression support",
            "risk_tier": "High",
            "human_review_required": True,
        },
    ]
)

inventory.to_csv(PROCESSED_DIR / "ai_system_inventory_demo.csv", index=False)
inventory

## Governance Evidence Pack

The toolkit checks whether essential evidence has been provided before approval.

In [ ]:
evidence_items = [
    "System description",
    "Decision influence map",
    "Data dictionary",
    "Demographic coverage summary",
    "Fairness and performance test results",
    "Human oversight procedure",
    "Vendor model card",
    "DPIA summary",
    "Equality impact assessment",
    "Student notice and explanation text",
    "Appeal route",
    "Rollback or suspension plan",
    "Monitoring schedule",
    "Incident log",
]

rng = np.random.default_rng(21)
rows = []

for system_id in inventory["system_id"]:
    for item in evidence_items:
        if system_id == "AI-001" and item in [
            "Fairness and performance test results",
            "Appeal route",
            "Human oversight procedure",
        ]:
            status = rng.choice(["Partial", "Missing"], p=[0.65, 0.35])
        elif system_id in ["AI-005", "AI-006"]:
            status = rng.choice(["Provided", "Partial", "Missing"], p=[0.42, 0.38, 0.20])
        else:
            status = rng.choice(["Provided", "Partial", "Missing"], p=[0.62, 0.28, 0.10])

        rows.append({
            "system_id": system_id,
            "evidence_item": item,
            "status": status,
            "is_required_for_approval": True,
        })

evidence = pd.DataFrame(rows)
evidence.to_csv(PROCESSED_DIR / "governance_evidence_pack_demo.csv", index=False)
evidence.head(15)

In [ ]:
evidence_summary = (
    evidence.groupby(["system_id", "status"])
    .size()
    .unstack(fill_value=0)
    .reset_index()
)

evidence_summary

## Residual Risk Engine

Score convention:

- `inherent_risk_score`: higher means riskier
- `control_maturity_score`: higher means stronger controls
- `evidence_completion_score`: higher means stronger evidence
- `residual_risk_score`: higher means more remaining risk after controls

In [ ]:
def deployment_decision(residual_risk_score: float) -> str:
    if residual_risk_score < 2.0:
        return "Approve"
    if residual_risk_score < 2.8:
        return "Approve with monitoring"
    if residual_risk_score < 3.5:
        return "Pilot only"
    if residual_risk_score < 4.2:
        return "Pause deployment"
    return "Reject / prohibit"


inherent_risk_map = {
    "ai_detection_tool": 4.5,
    "student_chatbot": 3.1,
    "vle_engagement_analytics": 3.4,
    "career_service_ai": 3.0,
    "ai_assisted_marking_feedback": 4.2,
    "early_alert_prediction": 4.1,
}

risk_rows = []

for _, system in inventory.iterrows():
    system_id = system["system_id"]
    system_type = system["system_type"]
    evidence_subset = evidence[evidence["system_id"] == system_id]

    provided_rate = (evidence_subset["status"] == "Provided").mean()
    partial_rate = (evidence_subset["status"] == "Partial").mean()

    evidence_completion_score = round((provided_rate + 0.5 * partial_rate) * 5, 2)
    control_maturity_score = round(max(1.0, min(5.0, evidence_completion_score - 0.3)), 2)
    inherent_risk_score = inherent_risk_map.get(system_type, 3.0)

    control_reduction_factor = 0.50
    residual_risk_score = round(
        inherent_risk_score * (1 - (control_maturity_score / 5) * control_reduction_factor),
        2,
    )

    risk_rows.append({
        "system_id": system_id,
        "system_name": system["system_name"],
        "system_type": system_type,
        "risk_tier": system["risk_tier"],
        "inherent_risk_score": inherent_risk_score,
        "control_maturity_score": control_maturity_score,
        "evidence_completion_score": evidence_completion_score,
        "residual_risk_score": residual_risk_score,
        "deployment_recommendation": deployment_decision(residual_risk_score),
    })

risk_assessment = pd.DataFrame(risk_rows)
risk_assessment.to_csv(PROCESSED_DIR / "system_risk_assessment_demo.csv", index=False)
risk_assessment

In [ ]:
plot_df = risk_assessment.sort_values("residual_risk_score", ascending=False)

plt.figure(figsize=(10, 5))
plt.bar(plot_df["system_name"], plot_df["residual_risk_score"])
plt.title("Governance Risk Scoring Demo: Residual Risk by AI System")
plt.xlabel("AI system")
plt.ylabel("Residual risk score")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()